## INP

### 全部数据

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import norm

# ==========================================
# 1. 读取数据 
# ==========================================
df = pd.read_csv(r'D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv')
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
target_temps = [-20, -25, -30, -35]

# ==========================================
# 2. 绘图准备与参数设置
# ==========================================
# 设置全局字体格式
def configure_plot_style():
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 为四个季节分配颜色
season_colors = {
    'Spring': '#4DAF4A', # 绿色
    'Summer': '#FF6666', # 红色
    'Autumn': '#6688FF', # 蓝色
    'Winter': '#B0B0B0', # 灰色
}

# 2行2列排版，figsize调整为 13x8 更适合A4纸竖排
fig, axes = plt.subplots(2, 2, figsize=(13, 8), dpi=500) 

# 将 2x2 的二维坐标轴矩阵展平为一维列表，方便用 i 索引
axes_flat = axes.flatten() 

# 调整子图间的横向(wspace)和纵向(hspace)间距
plt.subplots_adjust(wspace=0.25, hspace=0.35) 

# ==========================================
# 全局计算 X 轴的合理范围
# ==========================================
valid_global_inp = df['N_INP(#/L)'][df['N_INP(#/L)'] > 0]
global_log_data = np.log10(valid_global_inp)
x_min_global = np.floor(global_log_data.min())
x_max_global = np.ceil(global_log_data.max())
fixed_bin_width = 0.25
bins_edges = np.arange(x_min_global, x_max_global + fixed_bin_width, fixed_bin_width)
x_fit = np.linspace(x_min_global, x_max_global, 100)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, t in enumerate(target_temps):
    # 使用展平后的 axes_flat
    ax = axes_flat[i] 
    df_temp = df[df['T_a(degC)'] == t]
    
    legend_handles = []
    legend_labels = []
    
    for season in seasons:
        df_season = df_temp[df_temp['Season'] == season]
        if df_season.empty:
            continue
            
        valid_inp = df_season['N_INP(#/L)'][df_season['N_INP(#/L)'] > 0]
        data_log = np.log10(valid_inp)

        if len(data_log) < 3: 
            continue
            
        color = season_colors[season]
        weights = np.ones_like(data_log) / len(data_log) * 100
        
        # 绘制直方图
        counts, bins, _ = ax.hist(data_log, bins=bins_edges, weights=weights,
                                  histtype='step', linewidth=3.5, alpha=0.6, 
                                  color=color, zorder=2)
        
        # 高斯拟合
        mu, std = norm.fit(data_log)
        pdf = norm.pdf(x_fit, mu, std)
        
        bin_width = bins[1] - bins[0]
        y_fit = pdf * bin_width * 100
        
        ax.plot(x_fit, y_fit, linestyle='--', color=color, linewidth=1.2, zorder=3)
        
        # 计算 R^2
        bin_centers = (bins[:-1] + bins[1:]) / 2
        expected_counts = norm.pdf(bin_centers, mu, std) * bin_width * 100
        ss_res = np.sum((counts - expected_counts) ** 2)
        ss_tot = np.sum((counts - np.mean(counts)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        r2 = max(0, r2) 
        
        p_hist = mlines.Line2D([], [], color=color, linewidth=3.5, alpha=0.6)
        legend_handles.append((p_hist))
        
        # 为了保证排版整齐，建议给season填充空格保持等长，例如使用 {season:<6}
        legend_labels.append(f"{season:<6} -- Gauss Fit (R$^2$ = {r2:.2f})")

    # ==========================================
    # 4. 子图细节美化
    # ==========================================
    ax.set_title(f'T = {t} ± 1°C', fontsize=14, pad=10)
    ax.set_xlabel('LOG (INP Concentration [L$^{-1}$])', fontsize=11)
    ax.set_ylabel('Relative frequency [%]', fontsize=11)

    ax.set_xlim(x_min_global, x_max_global)
    ax.set_ylim(0, 50)
    
    ax.tick_params(axis='both', which='major', direction='in', 
                   top=True, right=True, length=6, width=1.2, labelsize=10)
    ax.tick_params(axis='both', which='minor', direction='in', 
                   top=True, right=True, length=3, width=1)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    
    ax.text(-0.15, 1.02, f'({chr(97+i)})', transform=ax.transAxes, 
            fontsize=14, fontweight='bold', va='bottom', ha='left')
    
    if legend_handles:
        ax.legend(legend_handles, legend_labels, 
                  handler_map={tuple: HandlerTuple(ndivide=None, pad=0.5)},
                  fontsize=8, frameon=True, 
                  edgecolor='black', framealpha=1, borderpad=0.4, 
                  handlelength=3, handletextpad=0.5)


# 5. 保存并显示
# bbox_inches='tight' 可以自动裁剪掉多余的白边
#plt.savefig('INP_Distribution_Seasons.png', dpi=600, bbox_inches='tight')
plt.show()

### 仅含显著数据点

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import norm

# ==========================================
# 1. 读取数据 
# ==========================================
df = pd.read_csv(r'D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv')
df = df[df['Is_Significant'] == True].copy()
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
target_temps = [-20, -25, -30, -35]

# ==========================================
# 2. 绘图准备与参数设置
# ==========================================
# 设置全局字体格式
def configure_plot_style():
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 为四个季节分配颜色
season_colors = {
    'Spring': '#4DAF4A', # 绿色
    'Summer': '#FF6666', # 红色
    'Autumn': '#6688FF', # 蓝色
    'Winter': '#B0B0B0', # 灰色
}

# 2行2列排版，figsize调整为 13x8 更适合A4纸竖排
fig, axes = plt.subplots(2, 2, figsize=(13, 8), dpi=500) 

# 将 2x2 的二维坐标轴矩阵展平为一维列表，方便用 i 索引
axes_flat = axes.flatten() 

# 调整子图间的横向(wspace)和纵向(hspace)间距
plt.subplots_adjust(wspace=0.25, hspace=0.35) 

# ==========================================
# 全局计算 X 轴的合理范围
# ==========================================
valid_global_inp = df['N_INP(#/L)'][df['N_INP(#/L)'] > 0]
global_log_data = np.log10(valid_global_inp)
x_min_global = np.floor(global_log_data.min())
x_max_global = np.ceil(global_log_data.max())
fixed_bin_width = 0.25
bins_edges = np.arange(x_min_global, x_max_global + fixed_bin_width, fixed_bin_width)
x_fit = np.linspace(x_min_global, x_max_global, 100)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, t in enumerate(target_temps):
    # 使用展平后的 axes_flat
    ax = axes_flat[i] 
    df_temp = df[df['T_a(degC)'] == t]
    
    legend_handles = []
    legend_labels = []
    
    for season in seasons:
        df_season = df_temp[df_temp['Season'] == season]
        if df_season.empty:
            continue
            
        valid_inp = df_season['N_INP(#/L)'][df_season['N_INP(#/L)'] > 0]
        data_log = np.log10(valid_inp)

        if len(data_log) < 3: 
            continue
            
        color = season_colors[season]
        weights = np.ones_like(data_log) / len(data_log) * 100
        
        # 绘制直方图
        counts, bins, _ = ax.hist(data_log, bins=bins_edges, weights=weights,
                                  histtype='step', linewidth=3.5, alpha=0.6, 
                                  color=color, zorder=2)
        
        # 高斯拟合
        mu, std = norm.fit(data_log)
        pdf = norm.pdf(x_fit, mu, std)
        
        bin_width = bins[1] - bins[0]
        y_fit = pdf * bin_width * 100
        
        ax.plot(x_fit, y_fit, linestyle='--', color=color, linewidth=1.2, zorder=3)
        
        # 计算 R^2
        bin_centers = (bins[:-1] + bins[1:]) / 2
        expected_counts = norm.pdf(bin_centers, mu, std) * bin_width * 100
        ss_res = np.sum((counts - expected_counts) ** 2)
        ss_tot = np.sum((counts - np.mean(counts)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        r2 = max(0, r2) 
        
        p_hist = mlines.Line2D([], [], color=color, linewidth=3.5, alpha=0.6)
        legend_handles.append((p_hist))
        
        # 为了保证排版整齐，建议给season填充空格保持等长，例如使用 {season:<6}
        legend_labels.append(f"{season:<6} -- Gauss Fit (R$^2$ = {r2:.2f})")

    # ==========================================
    # 4. 子图细节美化
    # ==========================================
    ax.set_title(f'T = {t} ± 1°C', fontsize=14, pad=10)
    ax.set_xlabel('$log_{10}$ (INP Concentration [L$^{-1}$])', fontsize=11)
    ax.set_ylabel('Relative frequency [%]', fontsize=11)

    ax.set_xlim(-0.75, 3.5)
    ax.set_ylim(0, 65)
    
    ax.tick_params(axis='both', which='major', direction='in', 
                   top=True, right=True, length=6, width=1.2, labelsize=10)
    ax.tick_params(axis='both', which='minor', direction='in', 
                   top=True, right=True, length=3, width=1)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    
    ax.text(-0.15, 1.02, f'({chr(97+i)})', transform=ax.transAxes, 
            fontsize=14, fontweight='bold', va='bottom', ha='left')
    
    if legend_handles:
        ax.legend(legend_handles, legend_labels, 
                  handler_map={tuple: HandlerTuple(ndivide=None, pad=0.5)},
                  fontsize=8, frameon=True, 
                  edgecolor='black', framealpha=1, borderpad=0.4, 
                  handlelength=3, handletextpad=0.5)


# 5. 保存并显示
# bbox_inches='tight' 可以自动裁剪掉多余的白边
#plt.savefig(r'D:\Coding\master0_2025\Thesis\fig 1.png', dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
# 绘制-15degC下INP的相对频率分布
# 注意: -15degC下仅在冬季和春季有数据
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import norm

# ==========================================
# 1. 读取数据 
# ==========================================
df = pd.read_csv(r'D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv')
df = df[df['Is_Significant'] == True].copy()# 仅保留显著数据点
target_temp = -15
df_temp = df[df['T_a(degC)'] == target_temp]

# ==========================================
# 2. 绘图准备与参数设置
# ==========================================
# 设置全局字体格式
def configure_plot_style():
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()
# 为四个季节分配颜色
season_colors = {
    'Spring': '#4DAF4A', # 绿色
    'Summer': '#FF6666', # 红色
    'Autumn': '#6688FF', # 蓝色
    'Winter': '#B0B0B0', # 灰色
}

fig, ax = plt.subplots(figsize=(7, 5), dpi=500)

# ==========================================
# 全局计算 X 轴的合理范围
# ==========================================
valid_global_inp = df['N_INP(#/L)'][df['N_INP(#/L)'] > 0]
global_log_data = np.log10(valid_global_inp)
x_min_global = np.floor(global_log_data.min())
x_max_global = np.ceil(global_log_data.max())
fixed_bin_width = 0.25
bins_edges = np.arange(x_min_global, x_max_global + fixed_bin_width, fixed_bin_width)
x_fit = np.linspace(x_min_global, x_max_global, 100)

# ==========================================
# 3. 绘制直方图和高斯拟合
# ==========================================
legend_handles = []
legend_labels = []
for season in season_colors.keys():
    df_season = df_temp[df_temp['Season'] == season]
    if df_season.empty:
        continue
        
    valid_inp = df_season['N_INP(#/L)'][df_season['N_INP(#/L)'] > 0]
    data_log = np.log10(valid_inp)

    if len(data_log) < 3: 
        continue
        
    color = season_colors[season]
    weights = np.ones_like(data_log) / len(data_log) * 100
    
    # 绘制直方图
    counts, bins, _ = ax.hist(data_log, bins=bins_edges, weights=weights,
                              histtype='step', linewidth=3.5, alpha=0.6, 
                              color=color, zorder=2)
    
    # 高斯拟合
    mu, std = norm.fit(data_log)
    pdf = norm.pdf(x_fit, mu, std)
    
    bin_width = bins[1] - bins[0]
    y_fit = pdf * bin_width * 100
    
    ax.plot(x_fit, y_fit, linestyle='--', color=color, linewidth=1.2, zorder=3)
    
    # 计算 R^2
    bin_centers = (bins[:-1] + bins[1:]) / 2
    expected_counts = norm.pdf(bin_centers, mu, std) * bin_width * 100
    ss_res = np.sum((counts - expected_counts) ** 2)
    ss_tot = np.sum((counts - np.mean(counts)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    r2 = max(0, r2) 
    
    p_hist = mlines.Line2D([], [], color=color, linewidth=3.5, alpha=0.6)
    legend_handles.append((p_hist))
    
    legend_labels.append(f"{season:<6} -- Gauss Fit (R$^2$ = {r2:.2f})")

# ==========================================
# 4. 子图细节美化
# ==========================================
ax.set_title(f'T = {target_temp} ± 1°C', fontsize=14, pad=10)
ax.set_xlabel('$log_{10}$ (INP Concentration [L$^{-1}$])', fontsize=11)
ax.set_ylabel('Relative frequency [%]', fontsize=11)
ax.set_xlim(-0.75, 3.5)
ax.set_ylim(0, 65)
ax.tick_params(axis='both', which='major', direction='in', 
               top=True, right=True, length=6, width=1.2, labelsize=10)
ax.tick_params(axis='both', which='minor', direction='in',
                top=True, right=True, length=3, width=1)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
if legend_handles:
    ax.legend(legend_handles, legend_labels, 
              handler_map={tuple: HandlerTuple(ndivide=None, pad=0.5)},
              fontsize=8, frameon=True, 
              edgecolor='black', framealpha=1, borderpad=0.4, 
              handlelength=3, handletextpad=0.5)
# 5. 保存并显示
plt.savefig(r'D:\Coding\master0_2025\Thesis\fig S1.png', dpi=500, bbox_inches='tight')
plt.show()

## n_s

### 全部数据

In [ ]:
## 绘制`n_s`的概率密度分布

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import norm

# ==========================================
# 1. 读取数据 
# ==========================================
df = pd.read_csv(r'D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\INP+ns(v1.0.2).csv')
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
target_temps = [-20, -25, -30, -35]

# ==========================================
# 2. 绘图准备与参数设置
# ==========================================
# 设置全局字体格式
def configure_plot_style():
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()

# 为四个季节分配颜色
season_colors = {
    'Spring': '#4DAF4A', # 绿色
    'Summer': '#FF6666', # 红色
    'Autumn': '#6688FF', # 蓝色
    'Winter': '#B0B0B0', # 灰色
}

# 2行2列排版，figsize调整为 13x8 更适合A4纸竖排
fig, axes = plt.subplots(2, 2, figsize=(13, 8), dpi=500) 

# 将 2x2 的二维坐标轴矩阵展平为一维列表，方便用 i 索引
axes_flat = axes.flatten() 

# 调整子图间的横向(wspace)和纵向(hspace)间距
plt.subplots_adjust(wspace=0.25, hspace=0.35) 

# ==========================================
# 全局计算 X 轴的合理范围
# ==========================================
valid_global_inp = df['n_s'][df['n_s'] > 0]
global_log_data = np.log10(valid_global_inp)
x_min_global = np.floor(global_log_data.min())
x_max_global = np.ceil(global_log_data.max())
fixed_bin_width = 0.25  
bins_edges = np.arange(x_min_global, x_max_global + fixed_bin_width, fixed_bin_width)
x_fit = np.linspace(x_min_global, x_max_global, 100)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, t in enumerate(target_temps):
    # 使用展平后的 axes_flat
    ax = axes_flat[i] 
    df_temp = df[df['T_a(degC)'] == t]
    
    legend_handles = []
    legend_labels = []
    
    for season in seasons:
        df_season = df_temp[df_temp['Season'] == season]
        if df_season.empty:
            continue
            
        valid_inp = df_season['n_s'][df_season['n_s'] > 0]
        data_log = np.log10(valid_inp)
        
        if len(data_log) < 3: 
            continue
            
        color = season_colors[season]
        weights = np.ones_like(data_log) / len(data_log) * 100
        
        # 绘制直方图
        counts, bins, _ = ax.hist(data_log, bins=bins_edges, weights=weights,
                                  histtype='step', linewidth=3.5, alpha=0.6, 
                                  color=color, zorder=2)
        
        # 高斯拟合
        mu, std = norm.fit(data_log)
        pdf = norm.pdf(x_fit, mu, std)
        
        bin_width = bins[1] - bins[0]
        y_fit = pdf * bin_width * 100
        
        ax.plot(x_fit, y_fit, linestyle='--', color=color, linewidth=1.2, zorder=3)
        
        # 计算 R^2
        bin_centers = (bins[:-1] + bins[1:]) / 2
        expected_counts = norm.pdf(bin_centers, mu, std) * bin_width * 100
        ss_res = np.sum((counts - expected_counts) ** 2)
        ss_tot = np.sum((counts - np.mean(counts)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        r2 = max(0, r2) 
        
        p_hist = mlines.Line2D([], [], color=color, linewidth=3.5, alpha=0.6)
        legend_handles.append((p_hist))
        
        # 为了保证排版整齐，建议给season填充空格保持等长，例如使用 {season:<6}
        legend_labels.append(f"{season:<6} -- Gauss Fit (R$^2$ = {r2:.2f})")

    # ==========================================
    # 4. 子图细节美化
    # ==========================================
    ax.set_title(f'T = {t} ± 1°C', fontsize=14, pad=10)
    ax.set_xlabel('LOG (Surface Active Site Density [m$^{-2}$])', fontsize=11)
    ax.set_ylabel('Relative frequency [%]', fontsize=11)

    ax.set_xlim(x_min_global, x_max_global)
    ax.set_ylim(0, 50)
    
    ax.tick_params(axis='both', which='major', direction='in', 
                   top=True, right=True, length=6, width=1.2, labelsize=10)
    ax.tick_params(axis='both', which='minor', direction='in', 
                   top=True, right=True, length=3, width=1)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    
    ax.text(-0.15, 1.02, f'({chr(97+i)})', transform=ax.transAxes, 
            fontsize=14, fontweight='bold', va='bottom', ha='left')
    
    if legend_handles:
        ax.legend(legend_handles, legend_labels, 
                  handler_map={tuple: HandlerTuple(ndivide=None, pad=0.5)},
                  fontsize=8, frameon=True, 
                  edgecolor='black', framealpha=1, borderpad=0.4, 
                  handlelength=3, handletextpad=0.5)


# 5. 保存并显示
# bbox_inches='tight' 可以自动裁剪掉多余的白边
#plt.savefig('INP_Distribution_Seasons.png', dpi=600, bbox_inches='tight')
plt.show()

### 仅含显著数据点

In [ ]:
## 绘制`n_s`的概率密度分布

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import norm

# ==========================================
# 1. 读取数据 
# ==========================================
df = pd.read_csv(r'D:\Coding\Data\Lanzhou_aerosol\SMPS+APS\INP+ns(v1.0.2).csv')
df = df[df['Is_Significant'] == True].copy()
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
target_temps = [-20, -25, -30, -35]

# ==========================================
# 2. 绘图准备与参数设置
# ==========================================
# 设置全局字体格式
def configure_plot_style():
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] 
    plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman'

configure_plot_style()

# 为四个季节分配颜色
season_colors = {
    'Spring': '#4DAF4A', # 绿色
    'Summer': '#FF6666', # 红色
    'Autumn': '#6688FF', # 蓝色
    'Winter': '#B0B0B0', # 灰色
}

# 2行2列排版，figsize调整为 13x8 更适合A4纸竖排
fig, axes = plt.subplots(2, 2, figsize=(13, 8), dpi=500) 

# 将 2x2 的二维坐标轴矩阵展平为一维列表，方便用 i 索引
axes_flat = axes.flatten() 

# 调整子图间的横向(wspace)和纵向(hspace)间距
plt.subplots_adjust(wspace=0.25, hspace=0.35) 

# ==========================================
# 全局计算 X 轴的合理范围
# ==========================================
valid_global_inp = df['n_s'][df['n_s'] > 0]
global_log_data = np.log10(valid_global_inp)
x_min_global = np.floor(global_log_data.min())
x_max_global = np.ceil(global_log_data.max())
fixed_bin_width = 0.25  
bins_edges = np.arange(x_min_global, x_max_global + fixed_bin_width, fixed_bin_width)
x_fit = np.linspace(x_min_global, x_max_global, 100)

# ==========================================
# 3. 循环绘制每个子图
# ==========================================
for i, t in enumerate(target_temps):
    # 使用展平后的 axes_flat
    ax = axes_flat[i] 
    df_temp = df[df['T_a(degC)'] == t]
    
    legend_handles = []
    legend_labels = []
    
    for season in seasons:
        df_season = df_temp[df_temp['Season'] == season]
        if df_season.empty:
            continue
            
        valid_inp = df_season['n_s'][df_season['n_s'] > 0]
        data_log = np.log10(valid_inp)
        
        if len(data_log) < 3: 
            continue
            
        color = season_colors[season]
        weights = np.ones_like(data_log) / len(data_log) * 100
        
        # 绘制直方图
        counts, bins, _ = ax.hist(data_log, bins=bins_edges, weights=weights,
                                  histtype='step', linewidth=3.5, alpha=0.6, 
                                  color=color, zorder=2)
        
        # 高斯拟合
        mu, std = norm.fit(data_log)
        pdf = norm.pdf(x_fit, mu, std)
        
        bin_width = bins[1] - bins[0]
        y_fit = pdf * bin_width * 100
        
        ax.plot(x_fit, y_fit, linestyle='--', color=color, linewidth=1.2, zorder=3)
        
        # 计算 R^2
        bin_centers = (bins[:-1] + bins[1:]) / 2
        expected_counts = norm.pdf(bin_centers, mu, std) * bin_width * 100
        ss_res = np.sum((counts - expected_counts) ** 2)
        ss_tot = np.sum((counts - np.mean(counts)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        r2 = max(0, r2) 
        
        p_hist = mlines.Line2D([], [], color=color, linewidth=3.5, alpha=0.6)
        legend_handles.append((p_hist))
        
        # 为了保证排版整齐，建议给season填充空格保持等长，例如使用 {season:<6}
        legend_labels.append(f"{season:<6} -- Gauss Fit (R$^2$ = {r2:.2f})")

    # ==========================================
    # 4. 子图细节美化
    # ==========================================
    ax.set_title(f'T = {t} ± 1°C', fontsize=14, pad=10)
    ax.set_xlabel('LOG (Surface Active Site Density [m$^{-2}$])', fontsize=11)
    ax.set_ylabel('Relative frequency [%]', fontsize=11)

    ax.set_xlim(x_min_global, x_max_global)
    ax.set_ylim(0, 50)
    
    ax.tick_params(axis='both', which='major', direction='in', 
                   top=True, right=True, length=6, width=1.2, labelsize=10)
    ax.tick_params(axis='both', which='minor', direction='in', 
                   top=True, right=True, length=3, width=1)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    
    ax.text(-0.15, 1.02, f'({chr(97+i)})', transform=ax.transAxes, 
            fontsize=14, fontweight='bold', va='bottom', ha='left')
    
    if legend_handles:
        ax.legend(legend_handles, legend_labels, 
                  handler_map={tuple: HandlerTuple(ndivide=None, pad=0.5)},
                  fontsize=8, frameon=True, 
                  edgecolor='black', framealpha=1, borderpad=0.4, 
                  handlelength=3, handletextpad=0.5)


# 5. 保存并显示
# bbox_inches='tight' 可以自动裁剪掉多余的白边
#plt.savefig('INP_Distribution_Seasons.png', dpi=600, bbox_inches='tight')
plt.show()